In [ ]:
import pandas as pd
import numpy as np
import os

def limpar_e_tratar_dados(caminho_entrada, caminho_saida):
    """
    Função completa para carregar, limpar, tratar e salvar o dataset Steam.
    """
    print(f"Carregando dados de: {caminho_entrada}")
    try:
        df = pd.read_csv(caminho_entrada)
        print("Dataset carregado com sucesso!")
    except FileNotFoundError:
        print(f"Erro: O arquivo '{caminho_entrada}' não foi encontrado.")
        return None

    print("Iniciando processo de limpeza...")

    # Cópia para evitar aviso de SettingWithCopyWarning
    df_clean = df.copy()

    # Remove espaços extras dos nomes das colunas
    df_clean.columns = df_clean.columns.str.strip()

    print(f"Linhas após carregamento: {len(df_clean)}")

    # Remove linhas totalmente vazias
    df_clean.dropna(how='all', inplace=True)
    print(f"Linhas após remover linhas totalmente vazias: {len(df_clean)}")

    # Corrige tipos de dados e trata valores ausentes
    if 'release_date' in df_clean.columns:
        df_clean['release_date'] = pd.to_datetime(df_clean['release_date'], errors='coerce')
        print("- Coluna 'release_date' convertida para datetime.")

    for col in ['genres', 'categories', 'steamspy_tags']:
        if col in df_clean.columns:
            df_clean[col] = df_clean[col].fillna('').astype(str).str.split(';')
    print("- Colunas 'genres', 'categories', 'steamspy_tags' transformadas em listas.")

    # --- CORREÇÃO: Converter listas para string temporariamente para remover duplicatas ---
    cols_lista = [col for col in ['genres', 'categories', 'steamspy_tags'] if col in df_clean.columns]
    for col in cols_lista:
        df_clean[col + '_str'] = df_clean[col].apply(lambda x: ';'.join(x) if isinstance(x, list) else '')

    # Após remover duplicatas
    duplicatas = df_clean.duplicated(subset=[col + '_str' for col in cols_lista]).sum()
    if duplicatas > 0:
        df_clean = df_clean.drop_duplicates(subset=[col + '_str' for col in cols_lista])
        print(f"- {duplicatas} linhas duplicadas foram removidas.")
    print(f"Linhas após remover duplicatas: {len(df_clean)}")

    # Remover as colunas auxiliares
    df_clean.drop(columns=[col + '_str' for col in cols_lista], inplace=True)

    # Criação de colunas booleanas para plataformas
    if 'platforms' in df_clean.columns:
        df_clean['platform_windows'] = df_clean['platforms'].str.contains('windows', na=False)
        df_clean['platform_mac'] = df_clean['platforms'].str.contains('mac', na=False)
        df_clean['platform_linux'] = df_clean['platforms'].str.contains('linux', na=False)
        print("- Colunas de plataforma (windows, mac, linux) criadas.")

    # Remove colunas irrelevantes, se existirem
    colunas_irrelevantes = [col for col in ['url'] if col in df_clean.columns]
    if colunas_irrelevantes:
        df_clean.drop(columns=colunas_irrelevantes, inplace=True)

    # Salva o resultado
    diretorio_saida = os.path.dirname(caminho_saida)
    if diretorio_saida and not os.path.exists(diretorio_saida):
        os.makedirs(diretorio_saida)
    df_clean.to_csv(caminho_saida, index=False, encoding='utf-8')
    print(f"Processo concluído! Arquivo limpo salvo em: {caminho_saida}")

    return df_clean

# --- Execução da Função ---
# Define os caminhos de entrada e saída baseados na nova estrutura de pastas
caminho_dados_brutos = 'steam.csv'
caminho_dados_limpos = 'steam_cleaned.csv'

# Chama a função para realizar todo o processo
df_final = limpar_e_tratar_dados(caminho_dados_brutos, caminho_dados_limpos)

if df_final is not None:
    print("\nAmostra do DataFrame final:")
    try:
        display(df_final.head())
    except NameError:
        print(df_final.head())
    
    print("\nInformações do DataFrame final:")
    df_final.info()

Carregando dados de: steam.csv
Dataset carregado com sucesso!
Iniciando processo de limpeza...
- Coluna 'release_date' convertida para datetime.
- Colunas 'genres', 'categories', 'steamspy_tags' transformadas em listas.
- 7704 linhas duplicadas foram removidas.
- Colunas de plataforma (windows, mac, linux) criadas.
Processo concluído! Arquivo limpo salvo em: steam_cleaned.csv

Amostra do DataFrame final:


,appid,name,release_date,english,developer,publisher,platforms,required_age,categories,genres,...,achievements,positive_ratings,negative_ratings,average_playtime,median_playtime,owners,price,platform_windows,platform_mac,platform_linux
0,10,Counter-Strike,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Local Mult...",[Action],...,0,124534,3339,17612,317,10000000-20000000,7.19,True,True,True
2,30,Day of Defeat,2003-05-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Valve Anti-Cheat enabled]",[Action],...,0,3416,398,187,34,5000000-10000000,3.99,True,True,True
4,50,Half-Life: Opposing Force,1999-11-01,1,Gearbox Software,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Valve Anti-Cheat...",[Action],...,0,5250,288,624,415,5000000-10000000,3.99,True,True,True
5,60,Ricochet,2000-11-01,1,Valve,Valve,windows;mac;linux,0,"[Multi-player, Online Multi-Player, Valve Anti...",[Action],...,0,2758,684,175,10,5000000-10000000,3.99,True,True,True
6,70,Half-Life,1998-11-08,1,Valve,Valve,windows;mac;linux,0,"[Single-player, Multi-player, Online Multi-Pla...",[Action],...,0,27755,1100,1300,83,5000000-10000000,7.19,True,True,True



Informações do DataFrame final:
<class 'pandas.core.frame.DataFrame'>
Index: 19371 entries, 0 to 27072
Data columns (total 21 columns):
 #   Column            Non-Null Count  Dtype         
---  ------            --------------  -----         
 0   appid             19371 non-null  int64         
 1   name              19371 non-null  object        
 2   release_date      19371 non-null  datetime64[ns]
 3   english           19371 non-null  int64         
 4   developer         19370 non-null  object        
 5   publisher         19361 non-null  object        
 6   platforms         19371 non-null  object        
 7   required_age      19371 non-null  int64         
 8   categories        19371 non-null  object        
 9   genres            19371 non-null  object        
 10  steamspy_tags     19371 non-null  object        
 11  achievements      19371 non-null  int64         
 12  positive_ratings  19371 non-null  int64         
 13  negative_ratings  19371 non-null  int64         